# Addition and Subtraction of Quantum Registers in Qiskit

This notebook demonstrates two methods for adding and subtracting two $n$-qubit registers $a$ and $b$ such that the result is written into register $b$:

- **Ripple-carry (Cuccaro MAJ/UMA):** uses a carry $c$ (1 ancilla qubit). 
- **Draper QFT adder:** uses QFT and controlled phase rotations, without ancilla qubits.

# ADDITION


## 1) Ripple-carry adder

The idea is the same as in classical binary addition. Moving from the LSB to the MSB, the carry is propagated in the ancilla qubit $c$, which is initially in the state $|0\rangle$.

The Cuccaro adder repeats two blocks for each bit $i$:

- MAJ($a_i,b_i,c$) - computes the next carry $c_{i+1}$ (the majority of $a_i,b_i,c$) and passes, or “ripples,” it forward.  
- UMA($a_i,b_i,c$) - goes backward, restores $a_i$, and leaves the sum bit in register $b$.

### MAJ($a_i,b_i,c$) gates
1) $CX(c \rightarrow b_i)$  
2) $CX(c \rightarrow a_i)$
4) $CCX(a_i, b_i \rightarrow c)$  

### UMA($a_i,b_i,c$) gates
1) $CCX(a_i, b_i \rightarrow c)$  
2) $CX(c \rightarrow a_i)$  
3) $CX(a_i \rightarrow b_i)$  

### Full circuit with $n$ bits
1) Forward pass (LSB to MSB): MAJ($a_0,b_0,c$), …, MAJ($a_{n-1},b_{n-1},c$)  
2) Backward pass (MSB to LSB): UMA($a_{n-1},b_{n-1},c$), …, UMA($a_0,b_0,c$)

Result: $b$ contains $(a+b)\bmod 2^n$, $a$ remains unchanged, and $c$ contains the final carry (carry-out).

In [1]:
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit_aer import AerSimulator

sim = AerSimulator()

def maj(qc, a, b, c):
    qc.cx(c, b)
    qc.cx(c, a)
    qc.ccx(a, b, c)

def uma(qc, a, b, c):
    qc.ccx(a, b, c)
    qc.cx(c, a)
    qc.cx(a, b)

def ripple_add(qc, a, b, carry):
    n = len(a)
    # carry forward
    for i in range(n):
        maj(qc, a[i], b[i], carry)
    # carry backward + sum in register b
    for i in reversed(range(n)):
        uma(qc, a[i], b[i], carry)

# -------------------------------------------------
# Example 1
a1 = QuantumRegister(3, "a")
b1 = QuantumRegister(3, "b")
carry1 = QuantumRegister(1, "carry")
cb1 = ClassicalRegister(3, "cb")
qc1 = QuantumCircuit(a1, b1, carry1, cb1)

# a = 5 = 101
qc1.x(a1[0])
qc1.x(a1[2])

# b = 3 = 011
qc1.x(b1[0])
qc1.x(b1[1])

ripple_add(qc1, a1, b1, carry1[0])
qc1.measure(b1, cb1)

tqc1 = transpile(qc1, sim)
counts1 = sim.run(tqc1, shots=1024).result().get_counts()

print("Ripple-carry: a=5, b=3")
print("Sum: ", counts1)
print(qc1.draw("text"))

# -------------------------------------------------
# Example 2 (superposition):
a2 = QuantumRegister(3, "a")
b2 = QuantumRegister(3, "b")
carry2 = QuantumRegister(1, "carry")
cb2 = ClassicalRegister(3, "cb")
qc2 = QuantumCircuit(a2, b2, carry2, cb2)

# a = (|000>+|100>)/sqrt2
qc2.h(a2[2])

# b = 1 = 001
qc2.x(b2[0])

ripple_add(qc2, a2, b2, carry2[0])

qc2.measure(b2, cb2)

tqc2 = transpile(qc2, sim)
counts2 = sim.run(tqc2, shots=4096).result().get_counts()

print("Ripple-carry: a=(|000>+|100>)/sqrt2, b=1 ")
print("Sum: ", counts2)
print(qc2.draw("text"))

Ripple-carry: a=5, b=3
Sum:  {'000': 1024}
       ┌───┐     ┌───┐                                                       »
  a_0: ┤ X ├─────┤ X ├──■────────────────────────────────────────────────────»
       └───┘     └─┬─┘  │       ┌───┐                                        »
  a_1: ────────────┼────┼───────┤ X ├──■──────────────────────────────────■──»
       ┌───┐       │    │       └─┬─┘  │       ┌───┐          ┌───┐       │  »
  a_2: ┤ X ├───────┼────┼─────────┼────┼───────┤ X ├──■────■──┤ X ├──■────┼──»
       ├───┤┌───┐  │    │         │    │       └─┬─┘  │    │  └─┬─┘  │    │  »
  b_0: ┤ X ├┤ X ├──┼────■─────────┼────┼─────────┼────┼────┼────┼────┼────┼──»
       ├───┤└─┬─┘  │    │  ┌───┐  │    │         │    │    │    │    │    │  »
  b_1: ┤ X ├──┼────┼────┼──┤ X ├──┼────■─────────┼────┼────┼────┼────┼────■──»
       └───┘  │    │    │  └─┬─┘  │    │  ┌───┐  │    │    │    │  ┌─┴─┐  │  »
  b_2: ───────┼────┼────┼────┼────┼────┼──┤ X ├──┼────■────■────┼──┤ X ├──┼──»
         

## 2) Draper QFT adder

This method uses the Quantum Fourier Transform (QFT) on register $b$.

QFT maps $|b\rangle$ into phase space, where addition can be implemented using controlled phase rotations depending on the bits of $a$. After that, $\mathrm{QFT}^\dagger$ returns $b$ back into number form.

The goal is the same:

$b \leftarrow (a+b)\bmod 2^n,$ and $a$ remains unchanged.

### High-level steps
1) $\mathrm{QFT}$ on $b$  
2) controlled rotations between $a$ and $b$  
3) $\mathrm{QFT}^\dagger$ on $b$

In Python, this is implemented with **DraperQFTAdder(n, kind="fixed")**

In [2]:
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit.circuit.library import DraperQFTAdder
from qiskit_aer import AerSimulator

sim = AerSimulator()
adder = DraperQFTAdder(3, kind="fixed")

# -------------------------------------------------
# Example 1:
a1 = QuantumRegister(3, "a")
b1 = QuantumRegister(3, "b")
cb1 = ClassicalRegister(3, "cb")
qc1 = QuantumCircuit(a1, b1, cb1)

# a = 5 = 101
qc1.x(a1[0])
qc1.x(a1[2])

# b = 3 = 011
qc1.x(b1[0])
qc1.x(b1[1])

qc1.append(adder, [*a1, *b1])
qc1.measure(b1, cb1)

tqc1 = transpile(qc1, sim)
counts1 = sim.run(tqc1, shots=1024).result().get_counts()

print("Draper QFT: a=5 (101), b=3 (011)")
print("Result:", counts1)
print(qc1.draw("text"))

# -------------------------------------------------
# Example 2:

a2 = QuantumRegister(3, "a")
b2 = QuantumRegister(3, "b")
cb2 = ClassicalRegister(3, "cb")
qc2 = QuantumCircuit(a2, b2, cb2)

# a = 0 or 4 ((|000>+|100>)/sqrt2)
qc2.h(a2[2])

# b = 1 (001)
qc2.x(b2[0])

qc2.append(adder, [*a2, *b2])
qc2.measure(b2, cb2)

tqc2 = transpile(qc2, sim)
counts2 = sim.run(tqc2, shots=4096).result().get_counts()

print("Draper QFT: a=(|000>+|100>)/sqrt2, b=1 (001)")
print("Result:", counts2)
print(qc2.draw("text"))

Draper QFT: a=5 (101), b=3 (011)
Result: {'000': 1024}
      ┌───┐┌─────────────────┐         
 a_0: ┤ X ├┤0                ├─────────
      └───┘│                 │         
 a_1: ─────┤1                ├─────────
      ┌───┐│                 │         
 a_2: ┤ X ├┤2                ├─────────
      ├───┤│  DraperQFTAdder │┌─┐      
 b_0: ┤ X ├┤3                ├┤M├──────
      ├───┤│                 │└╥┘┌─┐   
 b_1: ┤ X ├┤4                ├─╫─┤M├───
      └───┘│                 │ ║ └╥┘┌─┐
 b_2: ─────┤5                ├─╫──╫─┤M├
           └─────────────────┘ ║  ║ └╥┘
cb: 3/═════════════════════════╩══╩══╩═
                               0  1  2 
Draper QFT: a=(|000>+|100>)/sqrt2, b=1 (001)
Result: {'001': 2076, '101': 2020}
           ┌─────────────────┐         
 a_0: ─────┤0                ├─────────
           │                 │         
 a_1: ─────┤1                ├─────────
      ┌───┐│                 │         
 a_2: ┤ H ├┤2                ├─────────
      ├───┤│  DraperQFTAd

# SUBTRACTION

## 1) Ripple-carry subtraction

Subtraction $b \leftarrow (b-a)\bmod 2^n$ is obtained by executing the same circuit as in addition, but in the opposite direction, taking the inverse.

In the Cuccaro case, all gates are $\mathrm{CX}$ and $\mathrm{CCX}$. Their inverses are the same:
- $\mathrm{CX}^{-1}=\mathrm{CX}$
- $\mathrm{CCX}^{-1}=\mathrm{CCX}$
  
Therefore, reversing means only reversing the order, and no phase changes are needed.

### Reversing MAJ and UMA

If in addition:

**MAJ($a_i,b_i,c$)** is:
1) $\mathrm{CX}(c \rightarrow b_i)$  
2) $\mathrm{CX}(c \rightarrow a_i)$  
3) $\mathrm{CCX}(a_i, b_i \rightarrow c)$

**UMA($a_i,b_i,c$)** is:
1) $\mathrm{CCX}(a_i, b_i \rightarrow c)$  
2) $\mathrm{CX}(c \rightarrow a_i)$  
3) $\mathrm{CX}(a_i \rightarrow b_i)$  

Then in subtraction:

**MAJ$^{-1}$** is:
1) $\mathrm{CCX}(a_i, b_i \rightarrow c)$  
2) $\mathrm{CX}(c \rightarrow a_i)$  
3) $\mathrm{CX}(c \rightarrow b_i)$  

**UMA$^{-1}$** is:
1) $\mathrm{CX}(a_i \rightarrow b_i)$  
2) $\mathrm{CX}(c \rightarrow a_i)$  
3) $\mathrm{CCX}(a_i, b_i \rightarrow c)$  


### Execution scheme for subtraction

1) execute **UMA$^{-1}$** from LSB to MSB  
   UMA$^{-1}$($a_0,b_0,c$), …, UMA$^{-1}$($a_{n-1},b_{n-1},c$)
2) execute **MAJ$^{-1}$** from MSB to LSB  
   MAJ$^{-1}$($a_{n-1},b_{n-1},c$), …, MAJ$^{-1}$($a_0,b_0,c$)

In [3]:
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit_aer import AerSimulator

sim = AerSimulator()

def maj_inv(qc, a, b, c):
    qc.ccx(a, b, c)
    qc.cx(c, a)
    qc.cx(c, b)

def uma_inv(qc, a, b, c):
    qc.cx(a, b)
    qc.cx(c, a)
    qc.ccx(a, b, c)

def ripple_sub(qc, a, b, carry):
    n = len(a)
    for i in range(n):
        uma_inv(qc, a[i], b[i], carry)
    for i in reversed(range(n)):
        maj_inv(qc, a[i], b[i], carry)

# -------------------------------------------------
# Example
a = QuantumRegister(3, "a")
b = QuantumRegister(3, "b")
carry = QuantumRegister(1, "carry")
cb = ClassicalRegister(3, "cb")
qc = QuantumCircuit(a, b, carry, cb)

# a = 5 (101)
qc.x(a[0]); qc.x(a[1])
# b = 3 (011)
qc.x(b[0]); qc.x(b[2])

ripple_sub(qc, a, b, carry[0])  

qc.measure(b, cb)

counts = sim.run(transpile(qc, sim), shots=1024).result().get_counts()
print("Ripple-carry subtraction: a=3 (101), b=5 (011)")
print("Result:", counts)
print(qc.draw("text"))

Ripple-carry subtraction: a=3 (101), b=5 (011)
Result: {'010': 1024}
       ┌───┐               ┌───┐                                             »
  a_0: ┤ X ├──■────────────┤ X ├──■──────────────────────────────────────────»
       ├───┤  │            └─┬─┘  │  ┌───┐                                   »
  a_1: ┤ X ├──┼────■─────────┼────┼──┤ X ├──■─────────────────────────────■──»
       └───┘  │    │         │    │  └─┬─┘  │  ┌───┐          ┌───┐       │  »
  a_2: ───────┼────┼────■────┼────┼────┼────┼──┤ X ├──■────■──┤ X ├───────┼──»
       ┌───┐┌─┴─┐  │    │    │    │    │    │  └─┬─┘  │    │  └─┬─┘       │  »
  b_0: ┤ X ├┤ X ├──┼────┼────┼────■────┼────┼────┼────┼────┼────┼─────────┼──»
       └───┘└───┘┌─┴─┐  │    │    │    │    │    │    │    │    │         │  »
  b_1: ──────────┤ X ├──┼────┼────┼────┼────■────┼────┼────┼────┼─────────■──»
       ┌───┐     └───┘┌─┴─┐  │    │    │    │    │    │    │    │  ┌───┐  │  »
  b_2: ┤ X ├──────────┤ X ├──┼────┼────┼────┼────┼────■────■──

## 2) DraperQFT subtraction

Subtraction with the Draper adder is also obtained by taking the inverse of the addition circuit.

Unlike ripple-carry, the Draper adder consists of QFT and controlled phase rotations, so when reversing it, the direction of the rotations must also be reversed (the sign of the angle changes).

The inverse circuit for Draper subtraction:
1) $\mathrm{QFT}$ on $b$
2) The same controlled rotations, but with the opposite angle sign (for example, $R_k(\theta) \rightarrow R_k(-\theta)$)
3) $\mathrm{QFT}^\dagger$ on $b$

In [4]:
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit.circuit.library import DraperQFTAdder
from qiskit_aer import AerSimulator

sim = AerSimulator()

adder = DraperQFTAdder(3, kind="fixed")

# Subtraction using the inverse
sub = adder.inverse()

# -------------------------------------------------
# Example:
a = QuantumRegister(3, "a")
b = QuantumRegister(3, "b")
cb = ClassicalRegister(3, "cb")
qc = QuantumCircuit(a, b, cb)

# a = 5 = 101
qc.x(a[0]) 
qc.x(a[2])

# b = 3 = 011
qc.x(b[0]) 
qc.x(b[1])

qc.append(sub, [*a, *b])    
qc.measure(b, cb)

counts = sim.run(transpile(qc, sim), shots=1024).result().get_counts()
print("Draper QFT subtraction: a=5, b=3")
print("Result:", counts)
print(qc.draw("text"))

Draper QFT atņemšana: a=5, b=3
Rezultāts: {'000': 1024}
      ┌───┐┌─────────────────┐         
 a_0: ┤ X ├┤0                ├─────────
      └───┘│                 │         
 a_1: ─────┤1                ├─────────
      ┌───┐│                 │         
 a_2: ┤ X ├┤2                ├─────────
      ├───┤│  DraperQFTAdder │┌─┐      
 b_0: ┤ X ├┤3                ├┤M├──────
      ├───┤│                 │└╥┘┌─┐   
 b_1: ┤ X ├┤4                ├─╫─┤M├───
      └───┘│                 │ ║ └╥┘┌─┐
 b_2: ─────┤5                ├─╫──╫─┤M├
           └─────────────────┘ ║  ║ └╥┘
cb: 3/═════════════════════════╩══╩══╩═
                               0  1  2 
